# Data Validation and Integrity Checks

Comprehensive validation of the PMC corpus data to ensure quality and consistency.

## Validation Areas:
- File system integrity
- Data schema validation
- Content quality checks
- Cross-reference validation
- Missing data detection
- Anomaly identification

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
from pathlib import Path
from glob import glob
import json
import warnings
from datetime import datetime
import re
warnings.filterwarnings('ignore')

# Validation results storage
validation_results = {
    'timestamp': datetime.now().isoformat(),
    'checks': {},
    'errors': [],
    'warnings': [],
    'summary': {}
}

def log_check(check_name, status, message, details=None):
    """Log validation check results"""
    validation_results['checks'][check_name] = {
        'status': status,
        'message': message,
        'details': details or {}
    }
    
    if status == 'ERROR':
        validation_results['errors'].append(f"{check_name}: {message}")
    elif status == 'WARNING':
        validation_results['warnings'].append(f"{check_name}: {message}")

print("🔍 Data validation framework initialized")
print(f"Validation started at: {validation_results['timestamp']}")

## 1. File System Integrity Checks

In [ ]:
print("🗂️ Checking file system integrity...")

# Expected directory structure
expected_dirs = [
    '../output',
    '../output/data',
    '.'
]

# Expected file patterns
expected_files = {
    'tsv_metadata': '../*.tsv',
    'stats_files': '../*_stats.txt',
    'parquet_corpus': '../output/data/pmc_filtered.parquet',
    'notebooks': './*.ipynb'
}

# Check directories
for dir_path in expected_dirs:
    path = Path(dir_path)
    if path.exists() and path.is_dir():
        log_check(f"dir_{path.name}", "PASS", f"Directory exists: {path}")
    else:
        log_check(f"dir_{path.name}", "ERROR", f"Missing directory: {path}")

# Check file patterns
for file_type, pattern in expected_files.items():
    files = glob(pattern)
    if files:
        log_check(f"files_{file_type}", "PASS", 
                 f"Found {len(files)} {file_type} files", 
                 {'count': len(files), 'files': [Path(f).name for f in files[:5]]})
    else:
        log_check(f"files_{file_type}", "WARNING", f"No {file_type} files found")

# Check file sizes and accessibility
for file_type, pattern in expected_files.items():
    files = glob(pattern)
    for file_path in files[:3]:  # Check first 3 files of each type
        path = Path(file_path)
        try:
            size_mb = path.stat().st_size / (1024 * 1024)
            log_check(f"file_access_{path.name}", "PASS", 
                     f"File accessible: {path.name} ({size_mb:.1f} MB)")
        except Exception as e:
            log_check(f"file_access_{path.name}", "ERROR", 
                     f"Cannot access file: {path.name} - {e}")

print("✅ File system integrity check complete")

## 2. Data Schema Validation

In [ ]:
print("📋 Validating data schemas...")

# TSV metadata schema validation
tsv_files = glob('../*.tsv')
if tsv_files:
    expected_columns = [
        'Article File', 'Article Citation', 'AccessionID', 
        'LastUpdated (YYYY-MM-DD HH:MM:SS)', 'PMID', 'License', 'Retracted'
    ]
    
    for tsv_file in tsv_files[:2]:  # Check first 2 TSV files
        try:
            df = pd.read_csv(tsv_file, sep='\t', nrows=10)
            missing_cols = set(expected_columns) - set(df.columns)
            extra_cols = set(df.columns) - set(expected_columns)
            
            if not missing_cols and not extra_cols:
                log_check(f"schema_tsv_{Path(tsv_file).name}", "PASS", 
                         "TSV schema matches expected format")
            else:
                details = {}
                if missing_cols:
                    details['missing_columns'] = list(missing_cols)
                if extra_cols:
                    details['extra_columns'] = list(extra_cols)
                
                log_check(f"schema_tsv_{Path(tsv_file).name}", "WARNING", 
                         "TSV schema deviations found", details)
                
        except Exception as e:
            log_check(f"schema_tsv_{Path(tsv_file).name}", "ERROR", 
                     f"Cannot validate TSV schema: {e}")

# Parquet corpus schema validation
corpus_file = Path('../output/data/pmc_filtered.parquet')
if corpus_file.exists():
    try:
        df_corpus = pl.read_parquet(corpus_file)
        
        expected_corpus_cols = ['text', 'section']
        optional_corpus_cols = ['english_confidence_values', 'pmcid']
        
        present_cols = df_corpus.columns
        missing_required = set(expected_corpus_cols) - set(present_cols)
        
        if not missing_required:
            log_check("schema_corpus_parquet", "PASS", 
                     "Corpus parquet has required columns",
                     {'columns': present_cols, 'shape': df_corpus.shape})
        else:
            log_check("schema_corpus_parquet", "ERROR", 
                     "Missing required columns in corpus",
                     {'missing': list(missing_required)})
            
        # Check data types
        dtypes = dict(df_corpus.dtypes)
        if 'text' in present_cols and not str(dtypes['text']).startswith('Utf8'):
            log_check("dtype_corpus_text", "WARNING", 
                     f"Text column has unexpected dtype: {dtypes['text']}")
        else:
            log_check("dtype_corpus_text", "PASS", "Text column has correct dtype")
            
    except Exception as e:
        log_check("schema_corpus_parquet", "ERROR", 
                 f"Cannot validate corpus schema: {e}")
else:
    log_check("schema_corpus_parquet", "WARNING", "Corpus parquet file not found")

print("✅ Data schema validation complete")

## 3. Content Quality Validation

In [ ]:
print("📝 Validating content quality...")

# Load and validate TSV content
tsv_files = glob('../*.tsv')
if tsv_files:
    combined_tsv = []
    for tsv_file in tsv_files:
        try:
            df = pd.read_csv(tsv_file, sep='\t')
            combined_tsv.append(df)
        except Exception as e:
            log_check(f"load_tsv_{Path(tsv_file).name}", "ERROR", f"Cannot load TSV: {e}")
    
    if combined_tsv:
        papers_df = pd.concat(combined_tsv, ignore_index=True)
        
        # Check for missing critical data
        missing_pmcids = papers_df['AccessionID'].isna().sum()
        missing_pmids = papers_df['PMID'].isna().sum()
        missing_citations = papers_df['Article Citation'].isna().sum()
        
        if missing_pmcids == 0:
            log_check("content_pmcids", "PASS", "No missing PMC IDs")
        else:
            log_check("content_pmcids", "WARNING", 
                     f"{missing_pmcids:,} missing PMC IDs ({missing_pmcids/len(papers_df)*100:.1f}%)")
        
        if missing_pmids < len(papers_df) * 0.1:  # Allow 10% missing PMIDs
            log_check("content_pmids", "PASS", f"PMIDs mostly complete ({missing_pmids:,} missing)")
        else:
            log_check("content_pmids", "WARNING", f"Many missing PMIDs: {missing_pmids:,}")
            
        # Check for duplicates
        duplicate_pmcs = papers_df['AccessionID'].duplicated().sum()
        if duplicate_pmcs == 0:
            log_check("content_duplicates", "PASS", "No duplicate PMC IDs found")
        else:
            log_check("content_duplicates", "WARNING", f"{duplicate_pmcs:,} duplicate PMC IDs")
        
        # Validate citation format
        valid_citations = papers_df['Article Citation'].str.contains(r'\d{4}', na=False).sum()
        citation_validity = valid_citations / len(papers_df) * 100
        
        if citation_validity > 95:
            log_check("content_citations", "PASS", 
                     f"Citations well-formatted ({citation_validity:.1f}% valid)")
        else:
            log_check("content_citations", "WARNING", 
                     f"Some citation format issues ({citation_validity:.1f}% valid)")

# Validate corpus content quality
corpus_file = Path('../output/data/pmc_filtered.parquet')
if corpus_file.exists():
    try:
        df_corpus = pl.read_parquet(corpus_file)
        corpus_pd = df_corpus.to_pandas()
        
        if 'text' in corpus_pd.columns:
            # Text length validation
            text_lengths = corpus_pd['text'].str.len()
            empty_texts = (text_lengths == 0).sum()
            very_short = (text_lengths < 10).sum()
            very_long = (text_lengths > 10000).sum()
            
            if empty_texts == 0:
                log_check("content_empty_texts", "PASS", "No empty text entries")
            else:
                log_check("content_empty_texts", "WARNING", f"{empty_texts:,} empty text entries")
            
            if very_short < len(corpus_pd) * 0.05:  # Allow 5% very short texts
                log_check("content_short_texts", "PASS", f"Acceptable short texts ({very_short:,})")
            else:
                log_check("content_short_texts", "WARNING", f"Many very short texts: {very_short:,}")
            
            # Check for obvious encoding issues
            encoding_issues = corpus_pd['text'].str.contains(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', na=False).sum()
            if encoding_issues == 0:
                log_check("content_encoding", "PASS", "No obvious encoding issues")
            else:
                log_check("content_encoding", "WARNING", f"{encoding_issues:,} texts with encoding issues")
        
        # English confidence validation (if available)
        if 'english_confidence_values' in corpus_pd.columns:
            conf_values = corpus_pd['english_confidence_values']
            invalid_conf = ((conf_values < 0) | (conf_values > 1)).sum()
            
            if invalid_conf == 0:
                log_check("content_confidence_range", "PASS", "All confidence values in valid range")
            else:
                log_check("content_confidence_range", "ERROR", 
                         f"{invalid_conf:,} confidence values outside [0,1] range")
            
            mean_conf = conf_values.mean()
            if mean_conf > 0.3:  # Reasonable threshold
                log_check("content_confidence_quality", "PASS", 
                         f"Good average English confidence ({mean_conf:.3f})")
            else:
                log_check("content_confidence_quality", "WARNING", 
                         f"Low average English confidence ({mean_conf:.3f})")
                
    except Exception as e:
        log_check("content_corpus_validation", "ERROR", f"Cannot validate corpus content: {e}")

print("✅ Content quality validation complete")

## 4. Cross-Reference Validation

In [ ]:
print("🔗 Performing cross-reference validation...")

# Cross-validate between TSV metadata and corpus data
tsv_files = glob('../*.tsv')
corpus_file = Path('../output/data/pmc_filtered.parquet')

if tsv_files and corpus_file.exists():
    try:
        # Load TSV data
        tsv_dfs = []
        for tsv_file in tsv_files:
            try:
                df = pd.read_csv(tsv_file, sep='\t')
                tsv_dfs.append(df)
            except:
                continue
        
        if tsv_dfs:
            papers_df = pd.concat(tsv_dfs, ignore_index=True)
            unique_pmcs_tsv = set(papers_df['AccessionID'].dropna())
            
            # Load corpus data
            corpus_df = pl.read_parquet(corpus_file).to_pandas()
            
            # Check if corpus has PMC ID information
            if 'pmcid' in corpus_df.columns:
                unique_pmcs_corpus = set(corpus_df['pmcid'].dropna())
                
                # Calculate overlap
                common_pmcs = unique_pmcs_tsv & unique_pmcs_corpus
                tsv_only = unique_pmcs_tsv - unique_pmcs_corpus
                corpus_only = unique_pmcs_corpus - unique_pmcs_tsv
                
                overlap_percent = len(common_pmcs) / len(unique_pmcs_tsv) * 100
                
                if overlap_percent > 80:
                    log_check("crossref_pmc_overlap", "PASS", 
                             f"Good PMC ID overlap ({overlap_percent:.1f}%)",
                             {'common': len(common_pmcs), 'tsv_only': len(tsv_only), 
                              'corpus_only': len(corpus_only)})
                else:
                    log_check("crossref_pmc_overlap", "WARNING", 
                             f"Low PMC ID overlap ({overlap_percent:.1f}%)",
                             {'common': len(common_pmcs), 'tsv_only': len(tsv_only), 
                              'corpus_only': len(corpus_only)})
            else:
                log_check("crossref_pmc_overlap", "INFO", 
                         "Corpus does not contain PMC ID information for cross-referencing")
            
            # Check temporal consistency
            if 'LastUpdated (YYYY-MM-DD HH:MM:SS)' in papers_df.columns:
                dates = pd.to_datetime(papers_df['LastUpdated (YYYY-MM-DD HH:MM:SS)'], errors='coerce')
                valid_dates = dates.dropna()
                
                if len(valid_dates) > 0:
                    date_range = (valid_dates.max() - valid_dates.min()).days
                    recent_papers = (valid_dates > '2020-01-01').sum()
                    
                    log_check("crossref_temporal_span", "PASS", 
                             f"Data spans {date_range} days, {recent_papers:,} recent papers",
                             {'earliest': str(valid_dates.min().date()), 
                              'latest': str(valid_dates.max().date())})
                else:
                    log_check("crossref_temporal_span", "WARNING", "No valid dates found")
                    
    except Exception as e:
        log_check("crossref_validation", "ERROR", f"Cross-reference validation failed: {e}")
else:
    log_check("crossref_validation", "INFO", "Insufficient data for cross-reference validation")

print("✅ Cross-reference validation complete")

## 5. Statistical Anomaly Detection

In [ ]:
print("📊 Detecting statistical anomalies...")

# Anomaly detection in corpus data
corpus_file = Path('../output/data/pmc_filtered.parquet')
if corpus_file.exists():
    try:
        df_corpus = pl.read_parquet(corpus_file).to_pandas()
        
        if 'text' in df_corpus.columns:
            text_lengths = df_corpus['text'].str.len()
            
            # Detect outliers using IQR method
            Q1 = text_lengths.quantile(0.25)
            Q3 = text_lengths.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outliers_low = (text_lengths < lower_bound).sum()
            outliers_high = (text_lengths > upper_bound).sum()
            
            outlier_percentage = (outliers_low + outliers_high) / len(df_corpus) * 100
            
            if outlier_percentage < 5:  # Less than 5% outliers is normal
                log_check("anomaly_text_length", "PASS", 
                         f"Normal text length distribution ({outlier_percentage:.2f}% outliers)",
                         {'outliers_low': outliers_low, 'outliers_high': outliers_high,
                          'length_stats': {'mean': text_lengths.mean(), 'std': text_lengths.std(),
                                         'median': text_lengths.median()}})
            else:
                log_check("anomaly_text_length", "WARNING", 
                         f"High proportion of text length outliers ({outlier_percentage:.2f}%)",
                         {'outliers_low': outliers_low, 'outliers_high': outliers_high})
        
        # Section distribution anomalies
        if 'section' in df_corpus.columns:
            section_counts = df_corpus['section'].value_counts()
            
            # Check for extremely dominant sections
            top_section_pct = section_counts.iloc[0] / len(df_corpus) * 100
            if top_section_pct > 50:
                log_check("anomaly_section_dominance", "WARNING", 
                         f"One section dominates corpus ({top_section_pct:.1f}%): {section_counts.index[0]}")
            else:
                log_check("anomaly_section_dominance", "PASS", 
                         f"Balanced section distribution (top: {top_section_pct:.1f}%)")
            
            # Check for unusual section names
            unusual_sections = section_counts[section_counts == 1]  # Sections with only 1 occurrence
            if len(unusual_sections) > len(section_counts) * 0.3:  # More than 30% are singletons
                log_check("anomaly_section_singletons", "WARNING", 
                         f"Many singleton sections ({len(unusual_sections):,}), possible data quality issue")
            else:
                log_check("anomaly_section_singletons", "PASS", 
                         f"Reasonable section diversity ({len(unusual_sections):,} singletons)")
                         
    except Exception as e:
        log_check("anomaly_detection", "ERROR", f"Anomaly detection failed: {e}")

# Journal publication anomalies
tsv_files = glob('../*.tsv')
if tsv_files:
    try:
        tsv_dfs = []
        for tsv_file in tsv_files:
            try:
                df = pd.read_csv(tsv_file, sep='\t')
                tsv_dfs.append(df)
            except:
                continue
        
        if tsv_dfs:
            papers_df = pd.concat(tsv_dfs, ignore_index=True)
            
            # Extract journals
            journals = papers_df["Article Citation"].str.split(".", expand=True)[0]
            journal_counts = journals.value_counts()
            
            # Check for journal concentration
            top5_pct = journal_counts.head(5).sum() / len(papers_df) * 100
            if top5_pct > 70:
                log_check("anomaly_journal_concentration", "WARNING", 
                         f"High journal concentration: top 5 journals account for {top5_pct:.1f}%")
            else:
                log_check("anomaly_journal_concentration", "PASS", 
                         f"Good journal diversity: top 5 journals account for {top5_pct:.1f}%")
            
            # Check for suspicious journal patterns
            suspicious_patterns = [
                r'^\s*$',  # Empty or whitespace-only
                r'^\d+$',  # Only digits
                r'^[^a-zA-Z]*$',  # No letters
            ]
            
            suspicious_count = 0
            for pattern in suspicious_patterns:
                suspicious_count += journals.str.contains(pattern, na=False).sum()
            
            if suspicious_count == 0:
                log_check("anomaly_journal_names", "PASS", "No suspicious journal name patterns")
            else:
                log_check("anomaly_journal_names", "WARNING", 
                         f"{suspicious_count:,} entries with suspicious journal patterns")
                         
    except Exception as e:
        log_check("anomaly_journal_analysis", "ERROR", f"Journal anomaly detection failed: {e}")

print("✅ Statistical anomaly detection complete")

## 6. Validation Report Generation

In [ ]:
print("📋 Generating validation report...")

# Count validation results by status
status_counts = {'PASS': 0, 'WARNING': 0, 'ERROR': 0, 'INFO': 0}
for check_name, check_result in validation_results['checks'].items():
    status = check_result['status']
    if status in status_counts:
        status_counts[status] += 1

validation_results['summary'] = {
    'total_checks': len(validation_results['checks']),
    'status_counts': status_counts,
    'success_rate': (status_counts['PASS'] / len(validation_results['checks']) * 100) if validation_results['checks'] else 0,
    'has_errors': len(validation_results['errors']) > 0,
    'has_warnings': len(validation_results['warnings']) > 0
}

# Print comprehensive validation report
print("\n" + "="*60)
print("🔍 DATA VALIDATION REPORT")
print("="*60)
print(f"Validation completed at: {validation_results['timestamp']}")
print(f"Total checks performed: {validation_results['summary']['total_checks']}")
print(f"Success rate: {validation_results['summary']['success_rate']:.1f}%")
print()

# Status summary
print("📊 CHECK RESULTS SUMMARY:")
for status, count in status_counts.items():
    if count > 0:
        icon = {'PASS': '✅', 'WARNING': '⚠️', 'ERROR': '❌', 'INFO': 'ℹ️'}[status]
        print(f"   {icon} {status}: {count} checks")
print()

# Detailed results by category
categories = {
    'File System': [k for k in validation_results['checks'].keys() if k.startswith(('dir_', 'files_', 'file_'))],
    'Data Schema': [k for k in validation_results['checks'].keys() if k.startswith('schema_')],
    'Content Quality': [k for k in validation_results['checks'].keys() if k.startswith('content_')],
    'Cross-Reference': [k for k in validation_results['checks'].keys() if k.startswith('crossref_')],
    'Anomaly Detection': [k for k in validation_results['checks'].keys() if k.startswith('anomaly_')]
}

for category, check_names in categories.items():
    if check_names:
        print(f"📁 {category.upper()}:")
        for check_name in check_names:
            check_result = validation_results['checks'][check_name]
            status = check_result['status']
            message = check_result['message']
            icon = {'PASS': '✅', 'WARNING': '⚠️', 'ERROR': '❌', 'INFO': 'ℹ️'}[status]
            print(f"   {icon} {message}")
        print()

# Errors and warnings summary
if validation_results['errors']:
    print("❌ CRITICAL ERRORS:")
    for error in validation_results['errors']:
        print(f"   • {error}")
    print()

if validation_results['warnings']:
    print("⚠️  WARNINGS:")
    for warning in validation_results['warnings']:
        print(f"   • {warning}")
    print()

# Overall assessment
if not validation_results['errors'] and len(validation_results['warnings']) <= 2:
    print("🎉 OVERALL ASSESSMENT: EXCELLENT")
    print("   Data quality is high with no critical issues.")
elif not validation_results['errors']:
    print("👍 OVERALL ASSESSMENT: GOOD")
    print("   Data quality is acceptable with minor warnings.")
elif len(validation_results['errors']) <= 2:
    print("⚠️  OVERALL ASSESSMENT: NEEDS ATTENTION")
    print("   Some issues need to be addressed before production use.")
else:
    print("❌ OVERALL ASSESSMENT: POOR")
    print("   Multiple critical issues require immediate attention.")

print("\n" + "="*60)

# Save detailed validation report
report_file = Path(f"validation_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
try:
    with open(report_file, 'w') as f:
        json.dump(validation_results, f, indent=2)
    print(f"📄 Detailed report saved to: {report_file}")
except Exception as e:
    print(f"Warning: Could not save report file: {e}")

print("\n✅ Data validation complete!")

## Validation Checklist

### ✅ Completed Validations
- **File System Integrity**: Directory structure and file accessibility
- **Data Schema**: Column structure and data types
- **Content Quality**: Missing data, duplicates, format validation
- **Cross-Reference**: Consistency between different data sources
- **Statistical Anomalies**: Outlier detection and distribution analysis

### 🔧 Recommended Actions
Based on validation results:
1. **Critical Errors**: Address immediately before using data
2. **Warnings**: Review and consider fixing for improved quality
3. **Regular Monitoring**: Re-run validation after data updates

### 📊 Validation Metrics
- Success rate indicates overall data health
- Error count shows critical issues requiring attention
- Warning count indicates areas for improvement

**Note**: This validation framework should be run regularly, especially after data updates or pipeline changes.